<a href="https://colab.research.google.com/github/suhaasteja/VOD_reviewer/blob/main/VOD_review.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Clash Royale VOD Reviewer

Clash Royale is a fast-paced competitive mobile game where players deploy troops
to destroy the opponent's towers — whoever captures the most towers before the
clock runs out wins.

The final 30 seconds are where games are decided. This notebook uses Perceptron
Mk1 to analyze that critical window, surfacing the key decisions that won or lost
the match so you can study and improve your gameplay.

**What it does:**
- **VOD analysis** — reviews the last 30s of a match and diagnoses the single
  defining play: the outplay that secured the win, or the mistake that cost the loss
- **Card tracking** — upload a reference image of any troop card and the model
  will track how many times it was deployed and the damage it dealt (powered by
  in-context learning — no retraining required)

In [3]:
%pip install --upgrade perceptron --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.3/65.3 kB 2.3 MB/s eta 0:00:00


## Grab last 30 seconds of Clash royale gameplay

In [4]:
VIDEO_URL = "https://raw.githubusercontent.com/suhaasteja/VOD_reviewer/main/content/clash-royale_last_30s.mp4"

!curl -L -so last_30_s.mp4 {VIDEO_URL}

Clash Royal final 30s gameplay

<video src="https://raw.githubusercontent.com/suhaasteja/VOD_reviewer/main/content/clash-royale_last_30s.mp4" controls width="200" muted loop>
  Your browser doesn't support the video tag.
</video>

## Configure the Perceptron client

In [6]:
import os
from google.colab import userdata

from perceptron import configure, question, video, image, perceive, text


api_key = userdata.get('PERCEPTRON_API_KEY')
if not api_key or api_key.startswith("<"):
    raise RuntimeError("Set PERCEPTRON_API_KEY or replace the placeholder in this cell.")

configure(
    provider="perceptron",
    model="perceptron-mk1",
    api_key=api_key,
)

VIDEO_PATH = "last_30_s.mp4"

In [7]:
VOD_PROMPT = """Analyze these final 30 seconds of Clash Royale gameplay. Evaluate my performance based on whether this clip is a win or a loss:

- IF A LOSS: Diagnose the single defining mistake that cost me the match. Evaluate my win-condition targeting, elixir/cycle management, and card placement. Suggest the exact alternative card sequence that would have secured a win or tie.
- IF A WIN: Identify the key outplay or macro decision that secured the victory. Spot any hidden inefficiencies, resource mismanagement, or structural vulnerabilities I still committed despite winning.

"""

In [ ]:
QUESTION = VOD_PROMPT

result = question(video(VIDEO_PATH), QUESTION, reasoning=True, expects="clip")

print("--- Reasoning ---")
print(result.reasoning or "(none)")
print("\n--- Answer ---")
print(result.text)

--- Reasoning ---
So, let's analyze this Clash Royale clip. First, check if it's a win or loss. The video shows "Match Over!" and then the winner screen with "Suhass" as the winner (since the blue side is Suhass, and the crown is on their side). So it's a win.

Now, identify the key outplay or macro decision. Let's look at the gameplay:

- The player (Suhass) is on the blue side. The opponent (juao) is red.
- In the final seconds, the opponent's king tower is still up, but the player's towers are holding. Wait, no—wait, the match ends with the player winning, so the opponent's towers must have been destroyed.

Looking at the timeline: The timer counts down, and at the end, "Match Over!" appears, then the winner screen shows Suhass as winner. So the player won.

Key outplay: Let's see the card plays. The player uses a combination of troops and spells. For example, in the later part, maybe a well-timed spell or troop placement that prevented the opponent from pushing.

Wait, the opponent

card specific analytics

In [14]:
EXAMPLE_IMAGE = "https://raw.githubusercontent.com/suhaasteja/VOD_reviewer/main/content/hog-rider-on-field.jpg"

!curl -L -so hog-rider-on-field.jpg {EXAMPLE_IMAGE}

In [15]:
EXAMPLE_IMAGE="hog-rider-on-field.jpg"
QUERY_VIDEO="last_30_s.mp4"

## Reference image for in-context learning

<img src="https://raw.githubusercontent.com/suhaasteja/VOD_reviewer/main/content/hog-rider-on-field.jpg" width="100" alt="Hog Rider">

In [27]:
@perceive(reasoning=True, expects="clip")
def check_card_info(example_image_path: str, query_video_path: str):
    return (
        image(example_image_path)
        + text("this card is called hog-rider from a game called clash royale")
        + video(query_video_path)
        + text("how many times was it played and how much opponents tower health amount did it reduce?")
    )

result = check_card_info(EXAMPLE_IMAGE, QUERY_VIDEO)

print("--- Reasoning ---")
print(result.reasoning or "(none)")
print("\n--- Answer ---")
print(result.text)

--- Reasoning ---
So, let's look at the video. First, the Hog Rider is played once in the gameplay. Now, check the opponent's tower health. The opponent's tower (the one with the red crown) starts with some health. Let's see the numbers. In the video, when the Hog Rider attacks, the tower's health goes from 1274 to 843. Wait, let's calculate the reduction: 1274 - 843 = 431. So the Hog Rider was played once, and it reduced the opponent's tower health by 431.

--- Answer ---
To determine how many times the Hog Rider was played and the health reduction on the opponent’s tower:  

1. **Number of times played**: In the video, the Hog Rider is deployed *once* during the match (visible as the character is placed on the battlefield).  

2. **Opponent’s tower health reduction**:  
   - The opponent’s tower (top - red - crowned structure) starts with a health value of **1274** (seen early in the gameplay).  
   - After the Hog Rider’s attack, the tower’s health drops to **843** (visible later in

## Inspect the returned clips

In [28]:
clips = result.clips or []
if not clips:
    print("No clips returned. Try rephrasing the question or pointing at a different moment.")
else:
    print(f"Returned {len(clips)} clip(s):")
    for idx, clip in enumerate(clips, start=1):
        ts = clip.timestamp
        if ts.until is None:
            window = f"moment at {ts.at:.2f}s"
        else:
            window = f"{ts.at:.2f}s - {ts.until:.2f}s"
        label = clip.mention or "(no mention)"
        print(f"  Clip {idx}: {window} - {label}")

No clips returned. Try rephrasing the question or pointing at a different moment.
